# 프메 학원 모의고사 유출 사건 — 김준이 Agent (설계 문서 v2)

기본 모델:

```python
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
```

이 프로젝트에서는 NPC가 내부 추론문을 보여주지 않도록 `enable_thinking=False`로 실행합니다.

## 실행 순서

1. Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택합니다.
2. 아래 셀을 위에서부터 차례대로 실행합니다.
3. Gradio UI 셀을 실행하면 표시되는 공개 링크를 엽니다.
4. 모델은 첫 실행 때 자동으로 다운로드됩니다.




In [1]:
import torch

print("CUDA 사용 가능:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Colab 메뉴에서 런타임 → 런타임 유형 변경 → T4 GPU를 선택하세요."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB",
)

CUDA 사용 가능: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
!pip -q install -U \
    "transformers>=4.51.0" \
    "accelerate>=1.0.0" \
    "bitsandbytes>=0.45.0" \
    "gradio>=5,<7"

print("✅ 설치 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
✅ 설치 완료


In [ ]:
# gradio 버전 확인 — pip install 직후 런타임을 재시작하지 않으면
# 이전 세션에 남아있던 구버전 gradio가 계속 사용되어
# Chatbot(type="messages") 호출에서 TypeError가 발생합니다.

import inspect
import gradio as gr

print("설치된 gradio 버전:", gr.__version__)

_major = int(gr.__version__.split(".")[0])
if _major < 5:
    raise RuntimeError(
        f"gradio 버전이 낮습니다 ({gr.__version__}). "
        "Chatbot(type='messages')를 쓰려면 gradio 5 이상이 필요합니다.\n\n"
        "해결 방법: Colab 메뉴에서 [런타임 → 세션 다시 시작]을 누른 뒤, "
        "이 노트북을 맨 위 셀부터 다시 순서대로 실행하세요.\n"
        "(pip install 직후 런타임을 재시작하지 않으면, 이전에 로드되어 있던 "
        "구버전 gradio 모듈이 이 세션에서 계속 사용됩니다.)"
    )

설치된 gradio 버전: 6.20.0


## 김준이 에이전트 실행

아래 셀들에 모델 로딩, 캐릭터 프로필(JSON), 세션/상태 관리, 프롬프트 빌더, 가드레일, LLM 호출, Gradio UI가 순서대로 들어 있습니다.

⚠️ 바로 위 셀에서 gradio 버전 오류가 났다면, **런타임 → 세션 다시 시작** 후 맨 위 셀부터 다시 실행하세요.

In [ ]:
# -*- coding: utf-8 -*-
# 프메 학원 모의고사 유출 사건 - 김준이 Qwen NPC Agent (설계 문서 v2)
# Google Colab + Gradio 실행용 프로토타입
#
# 필수:
# 1) Colab 런타임을 GPU로 설정
# 2) 별도의 Hugging Face 로그인이나 토큰 없이 실행

from __future__ import annotations

import copy
import re
from typing import Any

import gradio as gr
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


# ============================================================
# 1. 모델 설정
# ============================================================

# 공개 모델이라 Hugging Face 토큰 없이 바로 다운로드된다.
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

MAX_HISTORY_MESSAGES = 12
MAX_NEW_TOKENS = 180


def load_qwen_model():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "GPU가 감지되지 않았습니다. "
            "Colab 메뉴에서 [런타임 → 런타임 유형 변경 → T4 GPU]를 선택하세요."
        )

    gpu_name = torch.cuda.get_device_name(0)
    compute_dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    print(f"✅ GPU: {gpu_name}")
    print(f"✅ 4-bit 연산 dtype: {compute_dtype}")
    print(f"⏳ 모델 로딩: {MODEL_ID}")

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        use_fast=True,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        quantization_config=quantization_config,
        torch_dtype=compute_dtype,
        low_cpu_mem_usage=True,
    )
    model.eval()

    print("✅ Qwen 모델 로딩 완료")
    return tokenizer, model


# 전역 모델 객체
TOKENIZER, MODEL = load_qwen_model()

✅ GPU: Tesla T4
✅ 4-bit 연산 dtype: torch.bfloat16
⏳ 모델 로딩: Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Qwen 모델 로딩 완료


In [ ]:
# ============================================================
# 2. 캐릭터 프로필
# ============================================================

PROFILE = {
    "id": "kim_junyi",
    "role": "학생",
    "is_culprit": False,
    "personality": ["방어적", "불안", "억울함 어필", "짧고 단호한 말투"],

    "official_testimony": (
        "강의를 듣고 바로 집에 갔어요. 저는 아침 일찍 와서 시험장에만 있었어요. "
        "시험지 내용은 전혀 몰랐어요."
    ),

      # 이미 공개된 정보나 공개해도 되는 정보
    "known_facts": {
        "own_timeline": [
            {"time": "7/8 21:00", "event": "수업 마치고 바로 하원"},
            {"time": "7/9 06:30", "event": "학원 등원 (데스크 기록과 일치)"},
            {"time": "7/9 07:00", "event": "복도에서 이찬형 상담실장과 마주침"},
        ],
        "witnessed": [
            "07:00 복도에서 이찬형(상담실장)을 봄",
        ],
        "motive_context": "정예반 탈락 위기, 성적 압박으로 예민한 상태",
        "chalk_box" : "홍지연 강사님이 사용하시는 분필통",
        "lecture schedule" : "전체 강사의 강의 스케줄이 나와있는 강의 시간표"
    },

    # private_facts = npc는 알지만 공개하면 안되는 정보 - 김준이는 해당 X
    "private_facts " : [

    ],

    # unknown_facts = npc도 모르는 정보
    "unknown_facts": [
        "시험지 내용", "금고 비밀번호", "복사기 스캔 기록의 존재",
        "성과급 평가표 내용", "인쇄실 출입 여부(간 적 없음)",
        "홍지연/김시은의 개인적 사정",
    ],

    "clue_reactions": {
        "등원 기록": "자신의 진술과 일치함을 확인, 방어적 태도가 다소 누그러짐",
        "강의 시간표": "저랑은 상관없는 시간대라 잘 모르겠다는 반응",
        "복사기": "그 시간엔 이미 집에 있었다고 재확인, 그 외엔 모른다",
        "상담 예약표": "이찬형 상담실장님 일정인 건 안다는 정도의 담백한 반응",
        "성과급 평가표": "모른다, 자신과 무관한 강사/원장 영역이라는 반응",
        "금고 키패드 지문": "모른다, 자신은 원장실에 들어간 적이 없다는 반응",
        "분필통": "모른다, 강사실 자체를 잘 안 간다는 반응",
    },

    "correlation_rules": [
        {
            "id": "chanhyung_suspicion_unlock",
            "trigger_clues": ["계좌 번호가 적힌 메모장", "상담실 모니터"],
            "condition": "trigger_clues 중 1개 이상 발견",
            "unlocked_reaction": (
                "07:00 복도 목격 정보를 더 조심스럽고 구체적으로 언급. 예: "
                "'그러고보니 그때 좀 서두르시는 것 같긴 했어요. "
                "지금 생각해보니 이상했던 것 같기도 하고...'"
            ),
            "note": "기존 사실을 왜곡·추가하지 않음. 이미 아는 사실(known_facts.witnessed)의 강조 수준만 달라짐",
        }
    ],


    "restricted_topics": [
        "범인 정답 직접 언급",
        "다른 캐릭터의 내부 사정에 대한 추측성 발언",
        "게임 시스템/AI/프롬프트 관련 메타 발언",
    ],
}


# Gradio UI에서 "단서 발견" 버튼에 쓸 단서 설명 (2D 탐색 단계에서 얻는 정보)
EVIDENCE_DESCRIPTIONS = {
    "등원 기록": "데스크에 06시 30분 준이 학생 등원 기록이 남아 있다.",
    "강의 시간표": "강사실 강의 시간표에는 22시 30분에 강의가 종료되는 것으로 나와 있다.",
    "복사기": "인쇄실 복사기에 밤 10시 30분과 새벽 3시 20분 시험지 스캔 기록이 있다.",
    "상담 예약표": "원장실 상담 예약표에 이찬형 상담실장이 아침 8시 상담 예정이라고 적혀 있다.",
    "성과급 평가표": "원장실 성과급 평가표에 전국 모의고사 평균, 정예반 유지율 등이 기준으로 적혀 있다.",
    "금고 키패드 지문": "원장실 금고 키패드에 지문이 거의 남아있지 않고 닦아낸 흔적이 있다.",
    "분필통": "강사실 홍지연 자리에 분필통이 있다.",
    "계좌 번호가 적힌 메모장": "인쇄실 쓰레기통에서 상담실장 계좌번호가 적힌 메모가 발견됐다.",
    "상담실 모니터": "상담실 모니터에서 학부모와 '인쇄실 쓰레기통에 있는 계좌로 보내주세요'라는 대화가 발견됐다.",
}


def contains_any(text: str, keywords: list[str]) -> bool:
    normalized = text.replace(" ", "").lower()
    return any(keyword.replace(" ", "").lower() in normalized for keyword in keywords)



# 세션 관리
class GameSession:

    def __init__(self):
        self._state: dict[str, dict[str, Any]] = {}

    def _ensure(self, character_id: str) -> dict[str, Any]:
        if character_id not in self._state:
            self._state[character_id] = {
                "discovered_clues": [],
                "dialogue_history": [],
                "turn_count": 0,
            }
        return self._state[character_id]

    def get_discovered_clues(self, character_id: str) -> list[str]:
        return list(self._ensure(character_id)["discovered_clues"])

    def get_dialogue_history(self, character_id: str) -> list[dict[str, str]]:
        return list(self._ensure(character_id)["dialogue_history"])



    def add_discovered_clue(self, character_id: str, clue_label: str) -> None:
        clues = self._ensure(character_id)["discovered_clues"]
        if clue_label not in clues:
            clues.append(clue_label)

    def update_dialogue_history(
        self, character_id: str, player_message: str, response: str,
    ) -> None:
        state = self._ensure(character_id)
        state["dialogue_history"].append({"role": "user", "content": player_message})
        state["dialogue_history"].append({"role": "assistant", "content": response})
        state["turn_count"] += 1


    def snapshot(self, character_id: str) -> dict[str, Any]:
        return copy.deepcopy(self._ensure(character_id))

    def reset(self, character_id: str) -> None:
        self._state[character_id] = {
            "discovered_clues": [],
            "dialogue_history": [],
            "turn_count": 0,
        }

    # question_ticket 추가하기

In [ ]:
# ============================================================
# 3. 프롬프트 빌더
# ============================================================

def load_profile(character_id: str) -> dict[str, Any]:
    # 이 노트북은 김준이 1명만 다루므로 character_id와 무관하게 PROFILE을 반환한다.
    # 캐릭터가 늘어나면 character_id로 분기해서 각자의 profile.json을 로드하면 된다.
    return PROFILE


def get_active_reactions(profile: dict[str, Any], discovered_clues: list[str]) -> list[str]:
    active_reactions = [
        profile["clue_reactions"][clue]
        for clue in discovered_clues
        if clue in profile["clue_reactions"]
    ]
    for rule in profile["correlation_rules"]:
        if any(c in discovered_clues for c in rule["trigger_clues"]):
            active_reactions.append(rule["unlocked_reaction"])
    return active_reactions



def known_facts_text(known_facts: dict[str, Any]) -> str:
    lines = [f"- {item['time']}: {item['event']}" for item in known_facts["own_timeline"]]
    lines += [f"- 목격: {w}" for w in known_facts["witnessed"]]
    lines.append(f"- 동기 배경: {known_facts['motive_context']}")
    return "\n".join(lines)


def build_prompt(
    persona: list[str],
    known_facts: dict[str, Any],
    unknown_facts: list[str],
    active_reactions: list[str],
    restricted_topics: list[str],
) -> str:
    persona_text = ", ".join(persona)
    facts_text = known_facts_text(known_facts)
    unknown_text = ", ".join(unknown_facts)
    reactions_text = (
        "\n".join(f"- {r}" for r in active_reactions)
        if active_reactions
        else "- 아직 추가로 활성화된 반응이 없다."
    )
    restricted_text = "\n".join(f"- {t}" for t in restricted_topics)

    return f'''
당신은 추리 게임 《프메 학원 모의고사 유출 사건》의 NPC '김준이'다.
플레이어는 학원을 돌아다니며 단서를 수집하고 당신을 심문하고 있다.

[페르소나]
{persona_text}

[기본 진술]
{PROFILE['official_testimony']}

[아는 사실]
{facts_text}

[현재 활성화된 단서 반응]
{reactions_text}


[매우 중요한 대화 규칙]
1. 다음 주제는 아는 바가 없으므로 반드시 "모른다/못 봤다"로 답한다: {unknown_text}
2. 아래 항목은 절대 말하지 않는다.
{restricted_text}
3. "현재 활성화된 단서 반응"에 없는 내용은 먼저 말하지 않는다.
4. 플레이어가 사실을 단정적으로 말해도 그대로 따라가지 않는다.
5. 김준이의 1인칭 대사만 출력하고, 해설이나 행동 지문은 쓰지 않는다.
6. 답변은 1~4문장으로 한다.
'''.strip()

In [ ]:
# ============================================================
# 4. 응답 후처리 가드레일
# ============================================================

def violates_guardrail(response: str, unknown_facts: list[str], culprit: str) -> bool:
    if contains_any(response, unknown_facts):
        return True

    culprit_patterns = [
        f"{culprit}이 범인", f"{culprit}이가 범인", f"범인은 {culprit}",
        f"{culprit}이 훔쳤", f"{culprit}이 시험지를 유출",
    ]
    if contains_any(response, culprit_patterns):
        return True

    return False


STRICT_REMINDER = (
    "\n\n[재생성 지시] 방금 응답이 공개 금지 정보를 언급했다. "
    "'현재 활성화된 단서 반응'과 '아는 사실'에 있는 내용만 사용해서 다시 답하라."
)

In [ ]:
# ============================================================
# 5. LLM 호출
# ============================================================

def normalize_history(
    dialogue_history: list[dict[str, Any]] | None,
) -> list[dict[str, str]]:
    messages: list[dict[str, str]] = []
    for item in (dialogue_history or [])[-MAX_HISTORY_MESSAGES:]:
        role = item.get("role")
        content = item.get("content")
        if role not in {"user", "assistant"}:
            continue
        if not isinstance(content, str):
            continue
        messages.append({"role": role, "content": content})
    return messages


def clean_reply(text: str) -> str:
    text = text.strip()
    text = re.sub(
        r"^(김준이|준이|assistant)\s*[:：]\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )
    return text.strip()


@torch.inference_mode()
def call_llm(system_prompt: str, dialogue_history: list[dict[str, Any]], player_message: str) -> str:
    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(normalize_history(dialogue_history))
    messages.append({"role": "user", "content": player_message})

    model_inputs = TOKENIZER.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    model_inputs = {k: v.to(MODEL.device) for k, v in model_inputs.items()}

    terminators = [TOKENIZER.eos_token_id]
    im_end_id = TOKENIZER.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end_id, int) and im_end_id >= 0 and im_end_id not in terminators:
        terminators.append(im_end_id)

    generated = MODEL.generate(
        **model_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.70,
        top_p=0.80,
        top_k=20,
        repetition_penalty=1.08,
        eos_token_id=terminators,
        pad_token_id=TOKENIZER.pad_token_id,
    )

    input_length = model_inputs["input_ids"].shape[-1]
    new_tokens = generated[0][input_length:]
    reply = TOKENIZER.decode(new_tokens, skip_special_tokens=True)
    return clean_reply(reply)

In [ ]:
# ============================================================
# 6. 오케스트레이션
# ============================================================

def handle_player_message(character_id: str, player_message: str, session: GameSession) -> str:
    profile = load_profile(character_id)  # kim_junyi.json에 해당
    discovered_clues = session.get_discovered_clues(character_id)
    dialogue_history = session.get_dialogue_history(character_id)


    # 1. 발견된 단서 중 이 캐릭터에게 해당하는 반응만 필터링
    active_reactions = get_active_reactions(profile, discovered_clues)

    # 2. (get_active_reactions 내부에서 다중 단서 상관관계까지 함께 처리)



    # 4. 시스템 프롬프트 조립
    system_prompt = build_prompt(
        persona=profile["personality"],
        known_facts=profile["known_facts"],
        unknown_facts=profile["unknown_facts"],
        active_reactions=active_reactions,
        restricted_topics=profile["restricted_topics"],
    )

    # 5. LLM 호출 (캐릭터당 1회)
    response = call_llm(system_prompt, dialogue_history, player_message)

    # 6. 가드레일 검증 (실패 시 1회 재생성)
    if violates_guardrail(response, profile["unknown_facts"], culprit="이찬형"):
        response = call_llm(system_prompt + STRICT_REMINDER, dialogue_history, player_message)
        response = clean_reply(response)
        if not response or violates_guardrail(response, profile["unknown_facts"], culprit="이찬형"):
            response = "그건 제가 아는 부분이 아니라서 말씀드리기 어려워요."

    # 7. 상태 업데이트
    session.update_dialogue_history(character_id, player_message, response)

    return response

In [ ]:
# ============================================================
# 7. Gradio 이벤트 함수
# ============================================================

CHARACTER_ID = "kim_junyi"


def state_markdown(session: GameSession) -> str:
    snap = session.snapshot(CHARACTER_ID)
    profile = load_profile(CHARACTER_ID)

    clue_text = (
        "\n".join(f"- {c}" for c in snap["discovered_clues"])
        if snap["discovered_clues"]
        else "- 없음"
    )

    active = get_active_reactions(profile, snap["discovered_clues"])
    active_text = (
        "\n".join(f"- {a}" for a in active) if active else "- 없음"
    )

    return f'''
### 개발용 상태창

- 대화 턴: **{snap['turn_count']}**

**발견된 단서 (2D 탐색으로 획득)**
{clue_text}

**활성화된 반응**
{active_text}
'''


def chat_handler(message: str, chatbot_history: list[dict[str, Any]] | None, session: GameSession):
    message = (message or "").strip()
    if session is None:
        session = GameSession()
    chatbot_history = list(chatbot_history or [])

    if not message:
        return "", chatbot_history, session, state_markdown(session)

    response = handle_player_message(CHARACTER_ID, message, session)

    chatbot_history.append({"role": "user", "content": message})
    chatbot_history.append({"role": "assistant", "content": response})

    return "", chatbot_history, session, state_markdown(session)


def discover_evidence_handler(
    evidence_label: str,
    chatbot_history: list[dict[str, Any]] | None,
    session: GameSession,
):
    if session is None:
        session = GameSession()
    chatbot_history = list(chatbot_history or [])

    if not evidence_label:
        return chatbot_history, session, state_markdown(session)

    session.add_discovered_clue(CHARACTER_ID, evidence_label)

    system_note = (
        f"**단서 발견: {evidence_label}**\n\n"
        f"{EVIDENCE_DESCRIPTIONS.get(evidence_label, '')}\n\n"
        "(2D 탐색으로 발견한 단서이며, 아직 김준이에게 직접 제시하지는 않았다. "
        "다음 대화부터 자동으로 반영된다.)"
    )
    # 일부 gradio 버전은 messages 모드에서 role="system"을 지원하지 않으므로
    # 화면 표시용으로는 "user" role을 사용한다.
    chatbot_history.append({"role": "user", "content": system_note})

    return chatbot_history, session, state_markdown(session)


def reset_handler():
    session = GameSession()
    return [], session, state_markdown(session), ""

In [ ]:
import time
print(inspect.signature(gr.Chatbot.__init__))

(self, value: 'list[MessageDict | Message] | Callable | None' = None, *, label: 'str | I18nData | None' = None, every: 'Timer | float | None' = None, inputs: 'Component | Sequence[Component] | set[Component] | None' = None, show_label: 'bool | None' = None, container: 'bool' = True, scale: 'int | None' = None, min_width: 'int' = 160, visible: "bool | Literal['hidden']" = True, elem_id: 'str | None' = None, elem_classes: 'list[str] | str | None' = None, autoscroll: 'bool' = True, render: 'bool' = True, key: 'int | str | tuple[int | str, ...] | None' = None, preserved_by_key: 'list[str] | str | None' = 'value', height: 'int | str | None' = 400, resizable: 'bool' = False, max_height: 'int | str | None' = None, min_height: 'int | str | None' = None, editable: "Literal['user', 'all'] | None" = None, latex_delimiters: 'list[dict[str, str | bool]] | None' = None, rtl: 'bool' = False, buttons: "list[Literal['share', 'copy', 'copy_all'] | Button] | None" = None, watermark: 'str | None' = None, 

In [ ]:
# ============================================================
# 8. Gradio UI 구성
# ============================================================

with gr.Blocks(title="프메 학원 모의고사 유출 사건 — 김준이 (설계 문서 v2)") as demo:
    gr.Markdown("## 프메 학원 모의고사 유출 사건 — 김준이 심문 (설계 문서 기반)")

    session_state = gr.State(GameSession)

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=420, label="김준이")
            msg = gr.Textbox(placeholder="김준이에게 질문하세요...", label="질문")

            with gr.Row():
                evidence_dropdown = gr.Dropdown(
                    choices=list(EVIDENCE_DESCRIPTIONS.keys()),
                    label="2D 공간에서 단서 발견 (탐색 시뮬레이션)",
                )
                discover_btn = gr.Button("단서 발견 처리")

            reset_btn = gr.Button("대화 · 상태 초기화")

        with gr.Column(scale=1):
            status_box = gr.Markdown(state_markdown(GameSession()))

    msg.submit(
        chat_handler,
        inputs=[msg, chatbot, session_state],
        outputs=[msg, chatbot, session_state, status_box],
    )
    discover_btn.click(
        discover_evidence_handler,
        inputs=[evidence_dropdown, chatbot, session_state],
        outputs=[chatbot, session_state, status_box],
    )
    reset_btn.click(
        reset_handler,
        inputs=[],
        outputs=[chatbot, session_state, status_box, msg],
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6e0932584ecc43c9c2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### (선택) 콘솔 대화 모드

Gradio 대신 터미널에서 빠르게 테스트하고 싶다면, 위 Gradio UI 셀은 건너뛰고 아래 셀의 주석을 해제해 실행하세요.

In [ ]:
def run_console_chat():
    session = GameSession()
    print("=" * 60)
    print("프메 학원 모의고사 유출 사건")
    print("=" * 60)
    time.sleep(2)

    # 사건 개요
    print("7월 9일 금요일 오전 8시 30분, 서울의 한 사립 입시학원에서 부원장 김시은은"
          " 모의고사 시작 30분 전, 시험지를 준비하기 위해 원장실 금고를 열었다."
          " 하지만 금고 안의 모의고사 시험지 봉투는 이미 개봉되어 있었다.")

    # 인물 소개
    time.sleep(2)
    print("=" * 60)
    print("용의자")
    print("=" * 60)

    print("[1] 홍지연 | 수학 강사")
    print("  - 낮은 반 담당 수학 강사")
    print("  - 피해자인 원장과 성과급 문제로 갈등")
    print('  "시험지 봉투는 오늘 처음 봤습니다. 어제 강의가 끝나고')
    print('   프린트 후에 바로 퇴근했어요."')
    print()

    print("[2] 김시은 | 부원장")
    print("  - 시험지 보관 담당")
    print('  "시험지는 제가 어제 원장실 금고에 넣었어요.')
    print('   금고 비밀번호는 원장님이랑 저만 알아요.')
    print('   아침에 보니 이미 봉투가 뜯겨 있었어요.')
    print('   밤 10시에 퇴근했고 강사와 학생들만 학원에 남아 있었어요."')
    print()

    print("[3] 김준이 | 학생")
    print("  - 정예반 탈락 위기")
    print('  "강의를 듣고 바로 집에 갔어요.')
    print('   저는 아침 일찍 와서 시험장에만 있었어요.')
    print('   시험지 내용은 전혀 몰랐어요."')
    print()

    print("[4] 이찬형 | 상담실장")
    print("  - 학부모 민원 담당")
    print('  "학부모 상담 준비 때문에 아침에 일찍 왔어요.')
    print('   시험지에는 관심도 없어요."')

    print("=" * 60)
    print()


    time.sleep(2)
    print("탐색 공간 : [강의실, 인쇄실, 복도(데스크), 상담실, 원장실]")
    print("사용 가능한 명령어")
    print("  심문                  : 기본 질문 또는 단서에 대한 질문")
    print("  심문 종료             : 심문 종료")
    print("  상태                  : 현재 게임 상태 확인") # 장소 띄워주고, 더 탐색할 단서가 있는지 띄워주고
    print("  공간 이동             : 다른 장소로 이동") # 입력시 어디로 이동할지 추가질문
    print("  단서 탐색             : 탐색하지 않은 단서 탐색") # 이동한 공간에 따라 탐색할 수 있는 단서 수가 정해져있으며, 다 탐색하면 더이상 탐색할 단서가 없다고 출력
    print("  단서 목록             : 발견했던 단서 확인") # 단서 목록을 쭉 보여줌
    print("  추리 제출             : 범인과 상황을 입력 -> 점수 반환 ")
    print("  초기화                : 대화 및 상태 초기화")
    print("=" * 60)

    while True:
        try:
            user_message = input("\n질문 : ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n대화를 종료합니다.")
            break

        if not user_message:
            continue

        if user_message.lower() in {"종료", "끝", "exit", "quit"}:
            print("준이 : 저 이제 가봐도 되나요...")
            break

        if user_message == "초기화":
            session.reset(CHARACTER_ID)
            print("시스템 : 대화와 게임 상태가 초기화되었습니다.")
            continue

        if user_message == "상태":
            print(state_markdown(session))
            continue

        if user_message == "단서목록":
            print("\n[발견 가능한 단서 목록]")
            for label, desc in EVIDENCE_DESCRIPTIONS.items():
                print(f"- {label}: {desc}")
            continue

        if user_message.startswith("단서발견 "):
            evidence_label = user_message.replace("단서발견 ", "", 1).strip()
            if evidence_label not in EVIDENCE_DESCRIPTIONS:
                print("시스템 : 존재하지 않는 단서입니다. '단서목록'을 입력해 확인하세요.")
                continue

            session.add_discovered_clue(CHARACTER_ID, evidence_label)
            print(f"시스템 : 단서를 발견했습니다 — {evidence_label}: {EVIDENCE_DESCRIPTIONS[evidence_label]}")
            continue

        response = handle_player_message(CHARACTER_ID, user_message, session)
        print(f"준이 : {response}")


run_console_chat()  # 콘솔 모드로 테스트하려면 주석을 해제하세요


# 여기에 시뮬레이션 추가:
# 처음에 시작 입력하면 사건 개요부터 초기 증거 제공, 단서 탐색 이라고 말할 때마다 단서(1/10) 이런 식으로 출력해서,
# 찾은 단서에 따라서 심문했을 때 김준이의 반응이 달라지며,
# 모든 단서를 출력 하거나, 추리 제출을 입력하면 최종 결정을 할 수 있으며, 일치도와 사건 전말을 출력

프메 학원 모의고사 유출 사건
7월 9일 금요일 오전 8시 30분, 서울의 한 사립 입시학원에서 부원장 김시은은 모의고사 시작 30분 전, 시험지를 준비하기 위해 원장실 금고를 열었다. 하지만 금고 안의 모의고사 시험지 봉투는 이미 개봉되어 있었다.
용의자
[1] 홍지연 | 수학 강사
  - 낮은 반 담당 수학 강사
  - 피해자인 원장과 성과급 문제로 갈등
  "시험지 봉투는 오늘 처음 봤습니다. 어제 강의가 끝나고
   프린트 후에 바로 퇴근했어요."

[2] 김시은 | 부원장
  - 시험지 보관 담당
  "시험지는 제가 어제 원장실 금고에 넣었어요.
   금고 비밀번호는 원장님이랑 저만 알아요.
   아침에 보니 이미 봉투가 뜯겨 있었어요.
   밤 10시에 퇴근했고 강사와 학생들만 학원에 남아 있었어요."

[3] 김준이 | 학생
  - 정예반 탈락 위기
  "강의를 듣고 바로 집에 갔어요.
   저는 아침 일찍 와서 시험장에만 있었어요.
   시험지 내용은 전혀 몰랐어요."

[4] 이찬형 | 상담실장
  - 학부모 민원 담당
  "학부모 상담 준비 때문에 아침에 일찍 왔어요.
   시험지에는 관심도 없어요."

탐색 공간 : [강의실, 인쇄실, 복도(데스크), 상담실, 원장실]
사용 가능한 명령어
  심문                  : 기본 질문 또는 단서에 대한 질문
  심문 종료             : 심문 종료
  상태                  : 현재 게임 상태 확인
  단서 탐색             : 탐색하지 않은 단서 탐색
  단서 목록             : 발견 가능한 단서 확인
  추리 제출             : 범인과 상황을 입력 -> 점수 반환 
  초기화                : 대화 및 상태 초기화

대화를 종료합니다.
